# BEAVERS: Time series for website
***

**_Author:_** Chus Casado Rodríguez<br>
**_Date:_** 01-09-2026<br>

**Introduction:**<br>
This script combines the daily reservoir operations with the daily basin meteorology and exports for each reservoir a time series file (Parquet) and a plot (HTML) to be shown in the website.

In [1]:
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import geopandas as gpd

import logging
logger = logging.getLogger(__name__)

from ocab.config import Config
import ocab.variables as VARS
import ocab.meteorology as METEO
from ocab.plots.reservoirs import plot_reservoir_timeseries, create_reservoir_html
from ocab.plots.utils import compute_climatology

## Configuration

In [2]:
cfg = Config('config_BEAVERS_v100.yml')

# LSTM model
model = None

# input paths
path_in = cfg.path_dataset / 'preprocessing' / 'timeseries'
# path results = cfg.path_dataset / 'results' / model if model is not None else None

# output paths
path_web = Path('../../docs')
path_layers = path_web / 'layers'
path_ts = path_web / 'timeseries' / 'reservoirs'
path_plots = path_ts / 'plots'
path_plots.mkdir(exist_ok=True, parents=True)
print(f'GeoJSON layers will be saved in:\t{path_layers}')
print(f'Parquet time series will be saved in:\t{path_ts}')
print(f'HTML plots will be saved in:\t\t{path_plots}')

# point layer
filename = 'reservoirs.geojson'

GeoJSON layers will be saved in:	../../docs/layers
Parquet time series will be saved in:	../../docs/timeseries/reservoirs
HTML plots will be saved in:		../../docs/timeseries/reservoirs/plots



## Create time series


In [3]:
# load reservoirs
reservoirs = gpd.read_file(cfg.path_gis / filename).set_index('id')

# add performance
if model is not None:
    performance_file = path_results / 'performance.geojson'
    if performance_file.is_file():
        # read performance values
        performance = gpd.read_file(performance_file)
        performance.rename(columns={'gauge_id': 'id'}, inplace=True)
        performance.set_index('id', inplace=True)
        performance = performance[~performance.index.duplicated(keep='first')]
        performance = performance.loc[performance.index.intersection(points.index)]
        # concatenate to points
        points = pd.concat([points, performance.drop(columns='geometry')], axis=1)
    else:
        logger.warning(f'The file {performance_file} does not exist')

In [4]:

# process timeseries for each station
for ID in tqdm(reservoirs.index, desc='reservoirs'):

    # reservoir operations
    try:
        resops = pd.read_parquet(path_in / 'resops' / f'{ID}.parquet')
        resops.rename(columns=VARS.RENAME, inplace=True, errors='ignore')
        # compute reservoir filling
        resops['filling'] = resops['storage_mcm'] / reservoirs.loc[ID, 'cap_mcm']
        # compute specific discharge (mm/day)
        for flow in ['inflow', 'outflow']:
            var = f'{flow}_cms'
            if var in resops.columns:
                resops[f'{flow}_mm'] = resops[var] / reservoirs.loc[ID, 'catch_skm'] * 86400 / 1000
        # clean errors in storage
        filling_error = resops['filling'] > 3
        if sum(filling_error) > 0:
            print(f'{ID} - No. days with storage larger than 3 times capacity: {sum(filling_error)}')
            resops.loc[filling_error, ['storage_mcm', 'filling']] = pd.NA
        # round
        resops = resops[resops.columns.intersection(VARS.DECIMALS)].round(VARS.DECIMALS)
    except Exception as e:
        print(f'Error loading reservoir operations for reservoir {ID}: {e}')
        continue

    # TODO: simulated discharge timeseries
    
    # meteo timeseries
    meteo = {}
    for dataset, label in METEO.DATASETS.items():
        try:
            df = pd.read_parquet(path_in / 'meteo' / dataset / f'{ID}.parquet').loc[ID]
            # correct index
            if dataset in METEO.OFFSET_HOURS:
                df.index = df.index.date + pd.Timedelta(hours=METEO.OFFSET_HOURS[dataset])
            df.index = pd.to_datetime(df.index)
            df.index.name = 'date'
            # rename and round variables
            df.rename(columns=VARS.RENAME, inplace=True, errors='ignore')
            df = df[df.columns.intersection(VARS.DECIMALS)].round(VARS.DECIMALS)
            # add dataset name to columns
            df.columns = [f'{col}_{label}' for col in df.columns]
            # meteo.append(df)
            meteo[label] = df
        except Exception as e:
            logger.error(f'Loading {dataset} meteo timeseries for station {ID}: {e}')

    # merge timeseries
    start = max(
        cfg.start, 
        min([df.first_valid_index() for df in meteo.values()]),
    )
    start_ts = max(
        start, 
        resops.first_valid_index() - pd.Timedelta(days=365)
    )
    start_html = max(
        start, 
        resops.first_valid_index()
    )
    end = min(
        cfg.end, 
        max([df.last_valid_index() for df in meteo.values()]),
        resops.last_valid_index()
    )
    ts = [resops.loc[start_ts:end]]
    if model is not None:
        ts.append(simulation)
    ts += [df.loc[start_ts:end] for df in meteo.values()]
    ts = pd.concat(ts, axis=1, sort=True)
 
    # export timeseries
    ts.to_parquet(path_ts / f'{ID}.parquet')
    
    # compute climatological values
    climatology = compute_climatology(ts)
    reservoirs.loc[ID, climatology.index] = climatology#.round(0)

    # add degree of regulation to attributes
    if reservoirs.loc[ID, 'outflow_mm'] > 0:
        dor_year = reservoirs.loc[ID, 'cap_mcm'] / (reservoirs.loc[ID, 'outflow_mm'] * reservoirs.loc[ID, 'catch_skm']) * 1e3
        reservoirs.loc[ID, 'dor_d'] = (dor_year * 365).round(0)
    else:
        reservoirs.loc[ID, 'dor_d'] = pd.NA

    try:
        # extract attributes
        attrs = reservoirs.loc[ID]

        # create time series plot
        title = '{0} - {1} - River {2} ({3})'.format(
            ID, 
            attrs['name'].title() if pd.notna(attrs['name']) else '', 
            attrs['river'].title() if pd.notna(attrs['river']) else '', 
            attrs['basin'].title()
        )
        # st, en = ts['filling'].first_valid_index(), ts['filling'].last_valid_index()
        fig = plot_reservoir_timeseries(
            ts.loc[start_html:end],
            attrs,
            title=title,
            save=True
        )

        # save plot as HTML
        create_reservoir_html(
            fig, 
            path=path_plots / f'{ID}.html', 
            start=start_html.strftime('%Y-%m-%d'), 
            end=end.strftime('%Y-%m-%d')
        )
    except Exception as e:
        print(f"The plot for time series {ID} couldn't be created:\n{e}")

# export updated points layer
reservoirs.to_file(path_layers / filename)

reservoirs:   0%|          | 0/374 [00:00<?, ?it/s]

1507 - No. days with storage larger than 3 times capacity: 1


/home/casadoj/Git/of_camels_and_beavers/.venv/lib/python3.12/site-packages/numpy/_core/function_base.py:163: RuntimeWarning: invalid value encountered in multiply
  y *= step
/home/casadoj/Git/of_camels_and_beavers/.venv/lib/python3.12/site-packages/numpy/_core/function_base.py:163: RuntimeWarning: invalid value encountered in multiply
  y *= step
/home/casadoj/Git/of_camels_and_beavers/.venv/lib/python3.12/site-packages/numpy/_core/function_base.py:163: RuntimeWarning: invalid value encountered in multiply
  y *= step


9843 - No. days with storage larger than 3 times capacity: 2


/home/casadoj/Git/of_camels_and_beavers/.venv/lib/python3.12/site-packages/numpy/_core/function_base.py:163: RuntimeWarning: invalid value encountered in multiply
  y *= step


10108 - No. days with storage larger than 3 times capacity: 11
10116 - No. days with storage larger than 3 times capacity: 1
